# Kaggle API setup
This notebook uses the repository Pixi environment and loads Kaggle credentials from `kaggleapi.txt`.

If `kaggleapi.txt` contains `username:key`, it will be parsed automatically. If it contains only the Kaggle key, set `KAGGLE_USERNAME` in your environment before running.

In [ ]:
import json
import os
from pathlib import Path

# Use kaggleapi.txt from the repo root for credentials
repo_root = Path('.')
key_file = repo_root / 'kaggleapi.txt'
if not key_file.exists():
    raise FileNotFoundError('kaggleapi.txt not found in the repository root. Add the key file and rerun.')

raw = key_file.read_text().strip()
username = None
key = None

if raw.startswith('{'):
    data = json.loads(raw)
    username = data.get('username') or data.get('user') or data.get('KAGGLE_USERNAME')
    key = data.get('key') or data.get('KAGGLE_KEY')
elif ':' in raw:
    username, key = raw.split(':', 1)
else:
    key = raw
    username = os.environ.get('KAGGLE_USERNAME')

if not key:
    raise ValueError('KAGGLE_KEY is required. Put the key in kaggleapi.txt or set KAGGLE_KEY in the environment.')

if username:
    os.environ['KAGGLE_USERNAME'] = username
os.environ['KAGGLE_KEY'] = key

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
token_path = kaggle_dir / 'kaggle.json'
token_path.write_text(json.dumps({'username': username, 'key': key}, indent=2))
try:
    token_path.chmod(0o600)
except Exception:
    pass
print(f'Kaggle credentials loaded from {key_file}')
if username:
    print(f'KAGGLE_USERNAME={username}')

In [ ]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()
print('Kaggle authentication succeeded')
print('Kaggle username:', os.environ.get('KAGGLE_USERNAME'))
print('Sample competitions:')
for comp in api.competitions_list(page=1, search='')[:5]:
    print('-', comp.ref)

In [ ]:
try:
    import kagglehub
    print('kagglehub is installed and available.')
except ImportError:
    print('kagglehub is not installed. Add it to pixi.toml and rebuild the environment.')